In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
import torch.quantization
import torchvision
from tqdm import tqdm
import time
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.datasets as datasets

## Fase 1

In [ ]:
def load_dataset(dataset_path, batch_size=32, test_split=0.2):
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=30),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(dataset_path, transform=train_transform)

    train_size = int((1 - test_split) * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

    test_dataset.dataset = datasets.ImageFolder(dataset_path, transform=test_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    num_classes = len(full_dataset.classes)
    class_names = full_dataset.classes

    print(f"Dataset caricato: {num_classes} classi, {len(train_dataset)} train, {len(test_dataset)} test")
    return train_loader, test_loader, num_classes, class_names

## Fase 2

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, num_classes):
        super(VisionTransformer, self).__init__()
        self.backbone = torchvision.models.vit_b_16(weights=None)
        self.backbone.heads.head = nn.Linear(768, num_classes)

    def forward(self, x):
        return self.backbone(x)

## Fase 3

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, teacher_model, temperature=2.0, alpha=0.5):
        super().__init__()
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, student_outputs, labels, inputs):
        with torch.no_grad():
            teacher_outputs = self.teacher(inputs)

        soft_teacher = torch.softmax(teacher_outputs / self.temperature, dim=1)
        soft_student = torch.log_softmax(student_outputs / self.temperature, dim=1)

        kd_loss = nn.KLDivLoss(reduction="batchmean")(soft_student, soft_teacher) * (self.temperature**2)
        ce_loss = self.ce_loss(student_outputs, labels)

        return self.alpha * ce_loss + (1 - self.alpha) * kd_loss

## Fase 4

In [ ]:
def test_model(model, test_loader, device, fp16=False):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            if fp16:
                inputs = inputs.half()
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100. * correct / total

def train_student(student, teacher, train_loader, test_loader, device, epochs=50, lr=0.001):
    criterion = DistillationLoss(teacher)
    optimizer = optim.Adam(student.parameters(), lr=lr)

    best_acc = 0.0
    for epoch in range(epochs):
        student.train()
        running_loss, correct, total = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = student(inputs)
            loss = criterion(outputs, targets, inputs)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            pbar.set_postfix({
                "Loss": f"{running_loss/len(train_loader):.4f}",
                "TrainAcc": f"{100.*correct/total:.2f}%"
            })

        acc = test_model(student, test_loader, device)
        print(f"Epoch {epoch+1}: TestAcc={acc:.2f}%")
        
        if acc > best_acc:
            best_acc = acc
            torch.save(student.state_dict(), "student_distilled_fp32.pth")

    print(f"✅ Training completato. BestAcc={best_acc:.2f}%")
    return student

## Fase 5


In [ ]:
def compress_and_export(student, test_loader, num_classes):
    device = torch.device("cpu")

    # PRUNING
    prune.l1_unstructured(student.classifier[1], name="weight", amount=0.3)
    prune.remove(student.classifier[1], "weight")
    torch.save(student.state_dict(), "student_pruned_fp32.pth")
    print("✅ Pruning eseguito e salvato.")

    # FP16 (half precision) - solo per GPU
    student_fp16 = student.half()
    torch.save(student_fp16.state_dict(), "student_fp16.pth")
    print("✅ FP16 salvato.")

    # Ripristina il modello in FP32 per i test su CPU
    student = student.float()

    # INT8 Quantization (Post Training)
    quantized_model = torch.quantization.quantize_dynamic(student.cpu(), {nn.Linear}, dtype=torch.qint8)
    torch.save(quantized_model.state_dict(), "student_int8.pth")
    print("✅ INT8 quantizzato e salvato.")

    # Test su CPU (solo FP32 e INT8)
    acc_fp32 = test_model(student.cpu(), test_loader, device="cpu")
    acc_int8 = test_model(quantized_model, test_loader, device="cpu")

    print(f"CPU Test Accuracies: FP32={acc_fp32:.2f}% | INT8={acc_int8:.2f}%")

    # Test su GPU FP16 (solo se disponibile)
    if torch.cuda.is_available():
        gpu = torch.device("cuda")
        student_fp16_gpu = student.half().to(gpu)
        acc_fp16 = test_model(student_fp16_gpu, test_loader, gpu, fp16=True)
        print(f"GPU FP16 Accuracy={acc_fp16:.2f}%")

## MAIN

In [ ]:
def main():
    dataset_path = "dataset"
    train_loader, test_loader, num_classes, class_names = load_dataset(dataset_path, batch_size=32, test_split=0.2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Teacher
    teacher = VisionTransformer(num_classes).to(device)
    checkpoint = torch.load("best_vision_transformer_model.pth", map_location=device, weights_only=True)

    if "backbone.heads.weight" in checkpoint:
        checkpoint["backbone.heads.head.weight"] = checkpoint.pop("backbone.heads.weight")
    if "backbone.heads.bias" in checkpoint:
        checkpoint["backbone.heads.head.bias"] = checkpoint.pop("backbone.heads.bias")

    # Handle checkpoint with Sequential Linear (head.0, head.2)
    if ("backbone.heads.head.0.weight" in checkpoint and
        "backbone.heads.head.2.weight" in checkpoint and
        "backbone.heads.head.0.bias" in checkpoint and
        "backbone.heads.head.2.bias" in checkpoint):
        checkpoint["backbone.heads.head.weight"] = checkpoint["backbone.heads.head.2.weight"]
        checkpoint["backbone.heads.head.bias"] = checkpoint["backbone.heads.head.2.bias"]

    teacher.load_state_dict(checkpoint, strict=False)
    teacher.eval()

    # Student
    student = torchvision.models.mobilenet_v2(weights=None, num_classes=num_classes).to(device)

    # Distillation training
    student = train_student(student, teacher, train_loader, test_loader, device, epochs=100, lr=0.001)

    # Compression & export
    compress_and_export(student, test_loader, num_classes)

main()

## Test su 100 immagini dal dataset (FP32, FP16 e INT8)

Questo esempio mostra come testare FP32 su CPU, FP16 su GPU e INT8 su CPU su 100 immagini prese dal test set.

In [ ]:
def show_misclassified_images(misclassified, subset, class_names, title, max_images=10):
    print(f"\n{title} - Mostra le prime {max_images} immagini sbagliate:")
    for err in misclassified[:max_images]:
        idx = err["index"]
        img, _ = subset[idx]
        img_np = img.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = std * img_np + mean
        img_np = np.clip(img_np, 0, 1)
        plt.imshow(img_np)
        plt.title(f"Idx: {idx}\nPred: {class_names[err['pred']]}\nTrue: {class_names[err['true']]}")
        plt.axis('off')
        plt.show()

In [ ]:
from torch.utils.data import Subset

dataset_path = "dataset"
train_loader, test_loader, num_classes, class_names = load_dataset(dataset_path, batch_size=32, test_split=0.2)

# Usa lo stesso test_loader e test_dataset già caricati
test_subset = Subset(test_loader.dataset, range(100))
subset_loader = torch.utils.data.DataLoader(test_subset, batch_size=32, shuffle=False)

# FP32 su CPU
student_fp32 = torchvision.models.mobilenet_v2(weights=None, num_classes=len(test_loader.dataset.dataset.classes))
try:
    student_fp32.load_state_dict(torch.load("student_distilled_fp32.pth", map_location="cpu", weights_only=True))
except FileNotFoundError:
    print("❌ File 'student_distilled_fp32.pth' non trovato. Esegui prima il training e la fase di export.")
    student_fp32 = None

if student_fp32 is not None:
    student_fp32.eval()

    misclassified_fp32 = []
    all_preds_fp32 = []
    all_targets_fp32 = []

    with torch.no_grad():
        for idx, (inputs, targets) in enumerate(subset_loader):
            outputs = student_fp32(inputs)
            _, preds = outputs.max(1)
            all_preds_fp32.extend(preds.cpu().numpy())
            all_targets_fp32.extend(targets.cpu().numpy())

            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    misclassified_fp32.append({
                        "index": idx * subset_loader.batch_size + i,
                        "pred": preds[i].item(),
                        "true": targets[i].item()
                    })

    acc_fp32 = sum([p == t for p, t in zip(all_preds_fp32, all_targets_fp32)]) / len(all_targets_fp32) * 100
    print(f"FP32 Accuracy su 100 immagini (CPU): {acc_fp32:.2f}%")
    print("FP32 - Errori (indice, predetta, reale):")
    for err in misclassified_fp32:
        print(f"Idx: {err['index']}, Pred: {class_names[err['pred']]}, True: {class_names[err['true']]}")

    #show_misclassified_images(misclassified_fp32, test_subset, class_names, "FP32", max_images=10)

    # INT8 su CPU
    quantized_model = torch.quantization.quantize_dynamic(student_fp32, {nn.Linear}, dtype=torch.qint8)
    misclassified_int8 = []
    all_preds_int8 = []
    all_targets_int8 = []

    with torch.no_grad():
        for idx, (inputs, targets) in enumerate(subset_loader):
            outputs = quantized_model(inputs)
            _, preds = outputs.max(1)
            all_preds_int8.extend(preds.cpu().numpy())
            all_targets_int8.extend(targets.cpu().numpy())
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    misclassified_int8.append({
                        "index": idx * subset_loader.batch_size + i,
                        "pred": preds[i].item(),
                        "true": targets[i].item()
                    })

    acc_int8 = sum([p == t for p, t in zip(all_preds_int8, all_targets_int8)]) / len(all_targets_int8) * 100
    print(f"INT8 Accuracy su 100 immagini (CPU): {acc_int8:.2f}%")
    print("INT8 - Errori (indice, predetta, reale):")
    for err in misclassified_int8:
        print(f"Idx: {err['index']}, Pred: {class_names[err['pred']]}, True: {class_names[err['true']]}")

    #show_misclassified_images(misclassified_int8, test_subset, class_names, "INT8", max_images=10)

    # FP16 su GPU
    if torch.cuda.is_available():
        student_fp16 = torchvision.models.mobilenet_v2(weights=None, num_classes=len(test_loader.dataset.dataset.classes)).half().to("cuda")
        student_fp16.load_state_dict(torch.load("student_fp16.pth", map_location="cuda", weights_only=True))
        student_fp16.eval()
        misclassified_fp16 = []
        all_preds_fp16 = []
        all_targets_fp16 = []

        with torch.no_grad():
            for idx, (inputs, targets) in enumerate(subset_loader):
                inputs = inputs.half().to("cuda")
                targets = targets.to("cuda")
                outputs = student_fp16(inputs)
                _, preds = outputs.max(1)
                all_preds_fp16.extend(preds.cpu().numpy())
                all_targets_fp16.extend(targets.cpu().numpy())
                
                for i in range(len(preds)):
                    if preds[i] != targets[i]:
                        misclassified_fp16.append({
                            "index": idx * subset_loader.batch_size + i,
                            "pred": preds[i].item(),
                            "true": targets[i].item()
                        })

        acc_fp16 = sum([p == t for p, t in zip(all_preds_fp16, all_targets_fp16)]) / len(all_targets_fp16) * 100

        print(f"FP16 Accuracy su 100 immagini (GPU): {acc_fp16:.2f}%")
        print("FP16 - Errori (indice, predetta, reale):")

        for err in misclassified_fp16:
            print(f"Idx: {err['index']}, Pred: {class_names[err['pred']]}, True: {class_names[err['true']]}")
        #show_misclassified_images(misclassified_fp16, test_subset, class_names, "FP16", max_images=10)
        
    else:
        print("FP16 test non disponibile: GPU non rilevata.")